# 01 - Datos y descriptivos

Este notebook lee el dataset oficial desde `data/`, muestra estadisticos descriptivos y genera las primeras figuras descriptivas en `figures/`.


## Configuracion del notebook

Esta celda define rutas internas, fechas de analisis y la funcion comun para guardar figuras en `figures/`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data"
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

DATA_START = "2010-01-01"
EVAL_START = "2015-01-02"
EVAL_END = "2024-12-27"


def savefig(name):
    plt.tight_layout()
    plt.savefig(FIGURES / f"{name}.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIGURES / f"{name}.pdf", bbox_inches="tight")
    plt.savefig(FIGURES / f"{name}.jpg", dpi=300, bbox_inches="tight")
    plt.show()


## Lectura y resumen del dataset

Esta celda carga el dataset oficial y muestra dimensiones, rango temporal y estadisticos basicos.


In [ ]:
dataset = pd.read_pickle(DATA / "dataset_tfg.pkl").sort_index()
print(dataset.shape)
print(dataset.index.min(), dataset.index.max())
display(dataset.head())
display(dataset[["target", "rp_lag_0", "vol_20", "vol_60"]].describe())


## Serie temporal de rentabilidades

Esta celda genera la figura de rentabilidad diaria de la cartera.


In [ ]:
df = dataset.loc[DATA_START:EVAL_END]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df.index, df["rp_lag_0"], lw=0.7, color="#1f4e79")
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Rentabilidad diaria de la cartera (2010-2024)")
ax.set_ylabel("Rentabilidad")
savefig("01_returns_time_series")


## Distribucion de perdidas

Esta celda genera el histograma de perdidas usado en el analisis descriptivo.


In [ ]:
losses = dataset.loc[EVAL_START:EVAL_END, "target"].dropna()
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(losses, bins=60, color="#6b8e23", edgecolor="white")
ax.set_title("Distribucion de perdidas de la cartera (2015-2024)")
ax.set_xlabel("Perdida")
ax.set_ylabel("Frecuencia")
savefig("02_loss_histogram")


## QQ plot normal

Esta celda compara graficamente la distribucion de perdidas frente a una normal teorica.


In [ ]:
losses = dataset.loc[EVAL_START:EVAL_END, "target"].dropna()
plt.figure(figsize=(5, 5))
stats.probplot(losses, dist="norm", plot=plt)
plt.title("QQ plot frente a normal (2015-2024)")
savefig("03_qq_normal")
